### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="houses",
    dataset_year="1990",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://lib.stat.cmu.edu/datasets/",
    download_description="""
We download the houses.zip from the lib.stat.cmu.edu repository and preprocess cadata.txt (strip the 27-line header text, remove leading whitespace, add a column header row).

mkdir -p local-data-warehouse/houses/ && wget -q -P /tmp https://lib.stat.cmu.edu/datasets/houses.zip && unzip -o /tmp/houses.zip -d /tmp && { echo "MedianHouseValue  MedianIncome  HousingMedianAge  TotalRooms  TotalBedrooms  Population  Households  Latitude  Longitude"; sed -n '28,$p' /tmp/cadata.txt | sed 's/^  //'; } > local-data-warehouse/houses/cadata_manual.txt && rm /tmp/houses.zip /tmp/cadata.txt
""",
    # References
    academic_reference_bibtex=r"""@article{pace1997sparse,
  title={Sparse spatial autoregressions},
  author={Pace, R Kelley and Barry, Ronald},
  journal={Statistics \\& Probability Letters},
  volume={33},
  number={3},
  pages={291--297},
  year={1997},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="pace1997sparse",
    license="Public",
    data_tags=["IID", "Spatial"],
    curation_comments="""
- The data contains rows whose values were capped artificially at a price of 500001. This creates a censored target, punishing model that learn to extrapolate from the data. We remove rows with this censored value in the target variable to obtain a more realistic task.
- We kept the latitude and longitude features as they are.
- We log scaled the target variable (with base e) as intended by the original task.
- Anomaly: As always, we randomly shuffle the data before uploading. If one does not randomly shuffle the data, there would be a distribution shift from the longitude and latitude based on the original order of data samples.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LnMedianHouseValue",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/cadata_manual.txt", index_col=False, sep="  ")

# Fix lat/longitude whitespace error from original data
df[["Latitude", "Longitude"]] = df["Latitude"].str.split(" ", expand=True)
df[["Latitude", "Longitude"]] = df[["Latitude", "Longitude"]].astype(float)

df = df[df["MedianHouseValue"] < 500001].reset_index(drop=True)


target_feature = "LnMedianHouseValue"

# Transform to log space as defined by original task
df[target_feature] = np.log(df["MedianHouseValue"])
df = df.drop(columns=["MedianHouseValue"])

# Spots a distribution shift in the dataset based on the order of the samples, removed by shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

/tmp/ipykernel_581938/3216028245.py:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(f"{dataset_mold.path}/cadata_manual.txt", index_col=False, sep="  ")


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 19,675
Columns: 9
Use sampling: False (sample size: 19,675)
Get row duplicates (staged, merged)...
Using top-8 columns for initial filtering: ['MedianIncome', 'TotalRooms', 'Population', 'TotalBedrooms', 'Households', 'Latitude', 'Longitude', 'HousingMedianAge']
Rows remaining as candidates after top-8 filter: 0 (of 19,675)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,MedianIncome,HousingMedianAge,TotalRooms,TotalBedrooms,Population,Households,Latitude,Longitude,LnMedianHouseValue
0,1.8357,24.0,2493.0,693.0,1420.0,643.0,32.80,-116.96,11.554067
1,4.2109,14.0,1946.0,463.0,1205.0,390.0,32.93,-117.14,12.050588
2,4.0481,52.0,1683.0,266.0,646.0,256.0,34.14,-117.29,11.485554
3,3.5380,21.0,1513.0,319.0,943.0,301.0,40.97,-124.01,11.539567
4,2.2000,34.0,2352.0,610.0,1127.0,592.0,38.62,-121.38,11.665647


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,MedianIncome,float64,0.0,0.0,12190.0,"3.125, 2.875, 4.125, 2.625, 3.875, 3.0, 3.375, 3.625, 4.0, 4.375"
1,HousingMedianAge,float64,0.0,0.0,52.0,"52.0, 36.0, 35.0, 16.0, 17.0, 34.0, 26.0, 33.0, 18.0, 25.0"
2,TotalRooms,float64,0.0,0.0,5793.0,"1527.0, 1582.0, 1613.0, 1607.0, 1703.0, 1717.0, 1722.0, 2053.0, 1787.0, 2127.0"
3,TotalBedrooms,float64,0.0,0.0,1910.0,"280.0, 331.0, 393.0, 343.0, 394.0, 348.0, 388.0, 314.0, 272.0, 309.0"
4,Population,float64,0.0,0.0,3868.0,"1052.0, 891.0, 1227.0, 782.0, 1005.0, 825.0, 872.0, 850.0, 1098.0, 781.0"
5,Households,float64,0.0,0.0,1793.0,"386.0, 306.0, 335.0, 429.0, 282.0, 297.0, 362.0, 375.0, 284.0, 330.0"
6,Latitude,float64,0.0,0.0,862.0,"34.08, 34.05, 34.09, 34.07, 34.02, 34.06, 34.04, 34.1, 34.03, 33.93"
7,Longitude,float64,0.0,0.0,842.0,"-118.31, -118.3, -118.29, -118.27, -118.28, -118.19, -118.35, -118.32, -118.36, -118.25"
8,LnMedianHouseValue,float64,0.0,0.0,3841.0,"11.8314, 11.9984, 11.6307, 12.1415, 12.3239, 12.7657, 11.3794, 12.5245, 11.9184, 12.0725"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
MedianIncome,19675.0,3.676717,1.570272,0.499900,15.000100
HousingMedianAge,19675.0,28.386277,12.509113,1.000000,52.000000
TotalRooms,19675.0,2619.763659,2181.348207,2.000000,39320.000000
TotalBedrooms,19675.0,539.653113,422.294861,2.000000,6445.000000
Population,19675.0,1440.812198,1143.648725,3.000000,35682.000000
Households,19675.0,501.186023,383.264636,2.000000,6082.000000
Latitude,19675.0,35.651780,2.149802,32.540000,41.950000
Longitude,19675.0,-119.563192,2.006108,-124.350000,-114.310000
LnMedianHouseValue,19675.0,12.033999,0.533308,9.615739,13.122363


In [7]:
# Categorical Feature Statistics
cat_stats

'No categorical/object features to summarize.'

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.295,-0.401,0.284,0.002,log,230567.5,5.200176e+15,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to houses/019d7368-4493-7f1a-a1e6-d784558ee09a
019d7368-4493-7f1a-a1e6-d784558ee09a
7a1da176bba68cd0aca86eab7ce4a0562eb1b2aa013ce0bd567ce8ff22a32966
